## **Maestría en Inteligencia Artificial Aplicada**
### **Curso: Inteligencia Artificial y Aprendizaje Automático**
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad de la Semana:**

### **Descomposición en Valores Singulares (SVD) y Sistemas de Recomendación**


**Esta Actividad deberá resolverse de manera individual.**

**Nombre y matrícula:**

*   Fernando Gomez Moreno - A00354097



# **Introducción**

In [51]:
# Agrega aquí todas las librerías y paquetes adicionales que requieras.

import numpy as np
import pandas as pd
import sklearn


from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity



* ### **Liga de datos de la UCI: "Restaurant & consumer data".**

https://archive.ics.uci.edu/dataset/232/restaurant+consumer+data

* ### **Del archivo comprimido "RCdata.zip" necesitamos solamente "rating_final.csv" y "geoplaces2.csv".**

* ### **NOTA: Al igual que con las variables numéricas, la información de variables con información de texto (strings) tiene sus propias técnicas de limpieza y transformación que podrás estudiar en cursos posteriores (en particular en el curso optativo de Procesamiento de Lenguaje Natural). No es el objetivo de este curso adentrarnos en dichas técnicas, pero sí requerimos hacer unos ajustes de limpieza para poder llevar a cabo la presente actividad. En particular, en la pregunta 2 te indicaré directamente cuál es el ajuste que hay que realizar para que el algoritmo de factorización SVD pueda funcionar más adelante.**

In [52]:
# Cargamos los archivos de la página de la UCI para construir
# nuestra matriz de utilidad:

data_rating = pd.read_csv("rating_final.csv", header='infer', sep=",")
data_geo = pd.read_csv("geoplaces2.csv", header='infer',  encoding='latin-1')

print(data_rating.shape, data_geo.shape)


(1161, 5) (130, 21)


# **Ejercicio - 1**

* ### **Explica cuál es el propósito del argumento "encoding" al cargar el segundo archivo. En particular, explica el error que te genera si omitimos dicho argumento y por qué el primer archivo no lo requiere.**

++++++++ Inicia la sección de agregar texto: +++++++++++


El argumento encoding se utiliza para indicar al intérprete de Python qué mapa de caracteres debe usar para traducir los bytes del archivo original a los caracteres de texto legibles.

Propósito: El valor 'latin-1' (también conocido como ISO-8859-1) permite que el programa procese correctamente caracteres especiales que no pertenecen al alfabeto inglés estándar, como las letras con acentos (á, é, í, ó, ú), la ñ o los símbolos de métrica que abundan en los nombres de establecimientos y direcciones en México.

Error generado al omitirlo: Si se omite, read_csv utiliza por defecto utf-8. Como el archivo geoplaces2.csv contiene caracteres extendidos que no siguen el estándar de codificación de UTF-8, se dispararía un error de tipo UnicodeDecodeError: 'utf-8' codec can't decode byte..., deteniendo la ejecución del código.

¿Por qué el primer archivo no lo requiere?: El archivo rating_final.csv contiene únicamente valores numéricos (IDs y calificaciones) y cadenas de texto simples (como el userID). Estos caracteres pertenecen al estándar ASCII, el cual es un subconjunto común tanto de UTF-8 como de Latin-1, por lo que puede leerse con la configuración por defecto sin causar conflictos de interpretación.


++++++++ Termina la sección de agregar texto. +++++++++++

In [53]:
# Veamos la lista de los nombres de los restaurantes de nuestros datos.
# Todos ellos son restaurantes ubicados en alguna ciudad del país de México:

data_geo['name'].values

array(['Kiku Cuernavaca', 'puesto de tacos', 'El Rincón de San Francisco',
       'little pizza Emilio Portes Gil', 'carnitas_mata',
       'Restaurant los Compadres', 'Taqueria EL amigo ', 'shi ro ie',
       'Pollo_Frito_Buenos_Aires', 'la Estrella de Dimas',
       'Restaurante 75', 'Abondance Restaurante Bar',
       'El angel Restaurante', 'Restaurante Pueblo Bonito',
       'Mcdonalds Parque Tangamanga', 'Tortas y hamburguesas el gordo',
       'Sirlone', 'rockabilly ', 'Unicols Pizza', 'TACOS EL GUERO',
       'Restaurant El Muladar de Calzada', 'La Posada del Virrey',
       'Restaurant and Bar and Clothesline Carlos N Charlies', 'KFC',
       'Giovannis', 'Restaurant Oriental Express', 'Mariscos Tia Licha',
       'cafe ambar', 'Restaurante la Gran Via', 'don burguers',
       'Restaurante y Pescaderia Tampico', 'Rincon del Bife',
       'La Fontana Pizza Restaurante and Cafe',
       'Restaurante la Estrella de Dima', 'El Rincon de San Francisco',
       'Preambulo Wifi Zone 

#### **Para que el algoritmo SVD funcione, cada registro (renglón) debe estar asociado a un restaurante con un nombre diferente.**

### **Algunos comentarios acerca de estos nombres. Para obtener la información de los siguientes comentarios en realidad requerimos algunas técnicas de procesamiento de lenguaje natural, pero como te comentaba, por el momento te las proporciono directamente para no distraernos con dichas técnicas de limpieza:**

*   ### **En la presente actividad generaremos un sistema de recomendación con base a la evaluación de varias características del servicio o tipo de comida de cada restaurante. Por el momento no tomaremos en cuenta la ubicación para generar nuestro sistema de recomendación, pero sí necesitamos aclarar ciertos puntos que menciono a continuación.**

* ### **La longitud y latitud es información que encuentras en el archivo geoplaces2.csv y son las coordenadas de la ubicación geográfica de cada restaurante. Se puede verificar que todas las coordenadas son diferentes, es decir, que todos los registros (renglones) contienen información de restaurantes ubicados en diferentes lugares de México.**

*   ### **Algunos restaurantes pertenezcan a una misma franquicia, como por ejemplo, Vips.**

*   ### **Para los fines de este ejercicio, cada registro de la base de datos lo consideraremos como un negocio diferente, independientemente de que pertenezcan a una misma franquicia. Así, en particular en el caso de Vips, existen tres restaurantes en esta lista capturados con los nombres: VIPS, Vips y vips. Por sus coordenadas de latitud y longitud sabemos que están ubicados en San Luis Potosí, Cuernavaca y Ciudad Victoria, tres ciudades diferentes de México. Como estos tres nombres están escritos de manera diferente, al momento de procesarlos con el algoritmo de SVD se estarán considerando como tres restaurantes diferentes. Por el momento es lo que queremos y por lo tanto no haremos ajustes en dichos nombres.**

*   ### **En el archivo existen más restaurantes pertenecientes a otras franquicias que han sido capturados también de manera diferente y nuevamente, así los dejaremos. Sin embargo, en el caso de 'Gorditas Dona Tota', dos de estos restaurantes están ubicados en Cd.Victoria y están escritos exactamente de la misma forma. Si los dejamos de esta manera, nuestro algoritmo SVD no funcionará, por ello, debes cambiar el nombre de uno de ellos como se te indica a continuación.**  

# **Ejercicio - 2**

* ### **Encuentra los índices de los dos restaurantes registrados exactamente con el mismo nombre de 'Gorditas Dona Tota' y cambia el nombre del que tiene el mayor índice en data_geo[name], al de  'Gorditas Dona Tota 2'.**

In [54]:
# Ejercicio-2

# ************* Inlcuye aquí tu código:*****************************


# Incluyamos en la siguiente variable la lista de todos los índices
# de restaurantes con exactamente el mismo nombre de 'Gorditas Dona Tota':

# Localizamos los índices de los registros con ese nombre exacto
indices_Dona_Tota = data_geo[data_geo['name'] == 'Gorditas Dona Tota'].index.tolist()

# Identificamos el índice mayor de la lista
indice_mayor = max(indices_Dona_Tota)

# Cambiamos el nombre en el DataFrame para ese índice específico
data_geo.loc[indice_mayor, 'name'] = 'Gorditas Dona Tota 2'




# *********** Aquí termina la sección de agregar código *************

print('Desplegando los nombres de los dos restaurantes con el ajuste:\n', data_geo.loc[indices_Dona_Tota, 'name'])


Desplegando los nombres de los dos restaurantes con el ajuste:
 89       Gorditas Dona Tota
118    Gorditas Dona Tota 2
Name: name, dtype: object


In [55]:
# Al momento tenemos los siguientes DataFrames. Del primer archivo
# tenemos la evaluación general, la de la comida y la del servicio:

data_rating.head(3)

,userID,placeID,rating,food_rating,service_rating
0,U1077,135085,2,2,2
1,U1077,135038,2,2,1
2,U1077,132825,2,2,2


In [56]:
# Y del segundo archivo obtenemos información diversa de cada restaurante:

data_geo.head(2).T

,0,1
placeID,134999,132825
latitude,18.915421,22.147392
longitude,-99.184871,-100.983092
the_geom_meter,0101000020957F000088568DE356715AC138C0A525FC46...,0101000020957F00001AD016568C4858C1243261274BA5...
name,Kiku Cuernavaca,puesto de tacos
address,Revolucion,esquina santos degollado y leon guzman
city,Cuernavaca,s.l.p.
state,Morelos,s.l.p.
country,Mexico,mexico
fax,?,?


# **Ejercicio - 3**

* ### **De cada uno de estos archivos selecciona y combina las variables adecuadas para obtener un nuevo DataFrame con 4 columnas: el ID de usuario (userID), el ID del restaurante (placeID), la calificación general (rating) y el nombre del restaurante (name). A este nuevo DataFrame llamarlo df_combinado.**

In [57]:
# Ejercicio 3:

# ************* Inlcuye aquí tu código:*****************************

# Seleccionamos las columnas necesarias de cada DataFrame y las combinamos usando 'placeID'
df_combinado = pd.merge(data_rating[['userID', 'placeID', 'rating']], 
                        data_geo[['placeID', 'name']], 
                        on='placeID')



# *********** Aquí termina la sección de agregar código *************


# Despleguemos la dimensión y los primeros renglones de este DataFrame:

print(df_combinado.shape)
df_combinado.head()

(1161, 4)


,userID,placeID,rating,name
0,U1077,135085,2,Tortas Locas Hipocampo
1,U1077,135038,2,Restaurant la Chalita
2,U1077,132825,2,puesto de tacos
3,U1077,135060,1,Restaurante Marisco Sam
4,U1068,135104,1,vips


# **Ejercicio - 4**

In [58]:
# Ejercicio 4:

#    Define la matriz de utilidad cuyos renglones sean los nombres de los
#    restaurantes, las columnas los IDs de los usuarios y las entradas la
#    evaluación general (rating). La llamaremos "UtMx_rating".


# ************* Inlcuye aquí tu código:*****************************

# Creamos la matriz de utilidad (pivoteando los datos)
# Renglones: nombres de restaurantes, Columnas: IDs de usuarios, Valores: rating
UtMx_rating = df_combinado.pivot_table(index='name', columns='userID', values='rating').fillna(0)



# *********** Aquí termina la sección de agregar código *************


print('Dimensión de la matriz de Utilidad:')
print('(restaurantes, usuarios) =', (UtMx_rating.shape))
UtMx_rating.head()

# Este paso es fundamental porque asocia las calificaciones numéricas de los usuarios con el nombre legible del restaurante, 
# lo cual nos permitirá interpretar mejor los resultados del sistema de recomendación más adelante.

Dimensión de la matriz de Utilidad:
(restaurantes, usuarios) = (130, 138)


userID,U1001,U1002,U1003,U1004,U1005,U1006,U1007,U1008,U1009,U1010,...,U1129,U1130,U1131,U1132,U1133,U1134,U1135,U1136,U1137,U1138
name,,,,,,,,,,,,,,,,,,,,,
Abondance Restaurante Bar,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Arrachela Grill,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cabana Huasteca,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0
Cafe Chaires,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Cafeteria cenidet,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### **Similaridades con la matriz obtenida de la Factorización SVD Truncada:**

*   ### **Aplicaremo la Factorización SVD con los 66 primeros valores singulares a la matriz de utilidad UtMx_rating.**

*   ### **El valor de 66 se seleccionó porque esta cantidad de valores singulares es suficiente para describir al menos el 95% de la varianza de la matriz de utilidad (información que mostraremos a continuación).**

*   ### **Recordemos que la factorización SVD de una matriz $A$ tiene la forma:** $A_{m\times n} = U_{m\times m}\Sigma_{m\times n}V_{n\times n}^T$

*   ### **Usaremos la función coseno como medida de similaridad**

*   ### **Usaremos el sistema de recomendación construído con SVD para buscar restaurantes similares al restaurante llamado  "tacos de barbacoa enfrente del Tec"**.

In [59]:
num_componentes=66  # componentes más significativas

# Obtengamos la matriz de factores latentes de los restaurantes:
SVD_rating = TruncatedSVD(n_components=num_componentes)  # inicializamos
matriz_latente_restaurantes = SVD_rating.fit_transform(UtMx_rating) # SVD truncada-ndarray

print('Dimensión Vectores latentes restaurantes:', matriz_latente_restaurantes.shape)
print('Dimensión Traspuesta Vectores latentes usuarios:', SVD_rating.components_.shape)

# Calculemos la variabilidad acumulada de las componentes utilizadas:
print('\nVariabilidad acumulada de las componentes utilizadas: %.3f' % SVD_rating.explained_variance_ratio_[0:num_componentes].sum())

# matriz de similaridad
sim_matrix = cosine_similarity(matriz_latente_restaurantes)

# Restaurante de referencia:
restaurante_de_referencia = "tacos de barbacoa enfrente del Tec"
nombres_rest = UtMx_rating.T.columns  # lista de los nombres de los restaurantes.
idx_rest = list(nombres_rest).index(restaurante_de_referencia) # índice del restaurante de referencia.
sim_vector = sim_matrix[idx_rest] # Vector de similaridad del restaurante de referencia contra todos.

# Buscando las similaridades positivas:
idx = (sim_vector>0)
mejores_sim_rate = list()
for i in range(len(nombres_rest[idx])):
  mejores_sim_rate.append((sim_vector[idx][i], nombres_rest[idx][i]))

print('\nTotal de similaridades positivas encontradas:', len(mejores_sim_rate))

# Las ordenamos de mayor a menor relevancia:
mejores_sim_rate_ordenadas = sorted(mejores_sim_rate, key=lambda x:x[0], reverse=True)

print('\nMejores recomendaciones con base al restaurante: \"%s\":' % restaurante_de_referencia)
mejores_sim_rate_ordenadas[1:11]   # omitimos el primero que es el mismo restaurante de referencia


Dimensión Vectores latentes restaurantes: (130, 66)
Dimensión Traspuesta Vectores latentes usuarios: (66, 138)

Variabilidad acumulada de las componentes utilizadas: 0.952

Total de similaridades positivas encontradas: 70

Mejores recomendaciones con base al restaurante: "tacos de barbacoa enfrente del Tec":


[(np.float64(0.9366203238862018), 'vips'),
 (np.float64(0.93467486846319), 'little pizza Emilio Portes Gil'),
 (np.float64(0.9258844925380693), 'tacos abi'),
 (np.float64(0.5305958276289047), 'Carreton de Flautas y Migadas'),
 (np.float64(0.5011327310369665), 'puesto de gorditas'),
 (np.float64(0.4802355940973822), 'Taqueria EL amigo '),
 (np.float64(0.36430017447140456), 'carnitas_mata'),
 (np.float64(0.26230381585368256), 'Little Cesarz'),
 (np.float64(0.23383834943284285), 'Gorditas Dona Tota 2'),
 (np.float64(0.18432707752510252), 'carnitas mata calle Emilio Portes Gil')]

# **Ejercicio - 5**

### **Sistema de Recomendación Híbrido**

* ### **Construyamos ahora un sistema de recomendación híbrido, el cual consistirá en conjuntar "matriz_latente_restaurantes" (que es un ndarray) obtenida previamente, con la matriz de factores adicionales (que también será un ndarray) formada por las columnas One-Hot-Encoder de los factores 'dress_code', 'accessibility', 'price' y 'franchise' del DataFrame data_geo.**

* ### **Los renglones de la matriz híbrida siguen representando a cada restaurante diferente, pero ahora con la información conjunta de los vectores latentes de la factorización SVD, junto con los vectores OneHotEncoding de los nuevos factores que incluídos. Esta combinación de información es una de las técnicas clásicas para contruir sistemas de recomendación más robustos.**



In [60]:
# Ejercicio 5:

# Define el DataFrame "df_fact_adicionales" formada por los factores 'dress_code',
# 'accessibility', 'price' y 'franchise'.
# Después genera la matriz (ndarray), "matriz_factores_adicionales_ohe", que se obtiene
# al transformar las columnas (variables categóricas nominales) de "df_fact_adicionales"
# mediante OneHotEncoding.
# Finalmente conjuntar horizontalmente la matriz "matriz_latente_restaurantes"
# con "matriz_factores_adicionales_ohe" para obtener la "matriz_hibrida" (ndarray).
# Toma en cuenta que "matriz_hibrida" estará formada por las columnas de
# "matriz_latente_restaurantes" seguida por las columnas de "mat_fact_adicionales_ohe".


# ************* Inlcuye aquí tu código:*****************************

# 1. Extraemos los factores adicionales asegurando que el orden coincida 
# con el de UtMx_rating (usando su índice de nombres)
df_fact_adicionales = data_geo.set_index('name').loc[UtMx_rating.index, ['dress_code', 'accessibility', 'price', 'franchise']]

# 2. Generamos la matriz One-Hot-Encoding (OHE)
# pd.get_dummies convierte las variables categóricas en columnas de 0s y 1s
matriz_factores_adicionales_ohe = pd.get_dummies(df_fact_adicionales).values


# 3. Conjuntamos horizontalmente los factores latentes de SVD con los factores OHE
# matriz_latente_restaurantes debe venir del paso anterior (SVD)
matriz_hibrida = np.concatenate((matriz_latente_restaurantes, matriz_factores_adicionales_ohe), axis=1)




# *********** Aquí termina la sección de agregar código *************


print('Dimensión de la matriz híbrida:')
matriz_hibrida.shape

Dimensión de la matriz híbrida:


(130, 77)

In [61]:
# Recordemos que "restaurante_de_referencia" es "tacos de barbacoa enfrente del Tec".
# Obtenengamos el índice de este restaurante:

idx = data_geo[data_geo['name'] == restaurante_de_referencia].index[0]

print('Restaurante de referencia:', restaurante_de_referencia)
print('Índice del restaurante de referencia:', idx)

Restaurante de referencia: tacos de barbacoa enfrente del Tec
Índice del restaurante de referencia: 111


# **Ejercicio - 6:**

* ### **Ahora que tienes la matriz híbrida, con dicha matriz obtener las similitudes (con respecto a la función coseno) del restaurante de referencia "tacos de barbacoa enfrente del Tec" con relación a todos los restaurantes. Deberás desplegar los primeros 10 restaurantes de mayor similitud (sin incluir al restaurante de referencia mismo), ordenados de mayor a menor similitud, y también desplegando el valor de similitud mismo.**

In [62]:
# Ejercicio 6

# ************* Inlcuye aquí tu código:*****************************

# Calcular la matriz de similitud de coseno para la matriz híbrida completa
sim_mat_hibrida = cosine_similarity(matriz_hibrida)

# Obtener el índice del restaurante de referencia dentro de los nombres
restaurante_referencia = "tacos de barbacoa enfrente del Tec"
nombres_rest = UtMx_rating.index
idx_ref = list(nombres_rest).index(restaurante_referencia)

# Extraer las similitudes del restaurante de referencia contra todos los demás
sim_vector_hibrido = sim_mat_hibrida[idx_ref]

# Generar una lista de tuplas (valor_similitud, nombre_restaurante)
sim_resultados = []
for i in range(len(nombres_rest)):
    sim_resultados.append((sim_vector_hibrido[i], nombres_rest[i]))

# Ordenar de mayor a menor basándonos en la similitud
sim_resultados_ordenados = sorted(sim_resultados, key=lambda x: x[0], reverse=True)

# Desplegar los primeros 10 (empezando en el índice 1 para omitir el restaurante mismo)
print(f'Mejores recomendaciones híbridas para: "{restaurante_referencia}":\n')
for sim, nombre in sim_resultados_ordenados[1:11]:
    print(f'{sim:.6f} : {nombre}')


# *********** Aquí termina la sección de agregar código *************

# utilice la función de similitud de coseno sobre la matriz híbrida que construi en el paso anterior. 
# Esto nos permitirá encontrar los restaurantes que más se parecen al de referencia, considerando tanto las calificaciones 
# de los usuarios como las características físicas del establecimiento.

# Explicación de lo realizado:
# - Similitud de Coseno: Calcule qué tan "cerca" están los vectores de la matriz híbrida entre sí. 
# Un valor cercano a 1 indica que los restaurantes son muy similares en sus características y en cómo los califican los usuarios.

# - Referencia: Localice la posición de "tacos de barbacoa enfrente del Tec" para extraer su fila de similitudes.

# - Ordenamiento: Organice los resultados de mayor a menor y mostre los 10 mejores resultados junto con su puntaje de similitud.


Mejores recomendaciones híbridas para: "tacos de barbacoa enfrente del Tec":

0.973395 : tacos abi
0.812641 : little pizza Emilio Portes Gil
0.769645 : Carnitas Mata  Calle 16 de Septiembre
0.736416 : Carreton de Flautas y Migadas
0.690801 : vips
0.669532 : puesto de gorditas
0.668209 : cafe ambar
0.653019 : Hamburguesas saul
0.633185 : sirloin stockade
0.599346 : Taqueria EL amigo 


# **Ejercicio - 7**

### **Incluye tus comentarios y conclusiones de la actividad.**

++++++++ Inicia la sección de agregar texto: +++++++++++


Comentarios y Conclusiones:

Potencia de la Reducción de Dimensionalidad (SVD):
A través de la técnica de Descomposición en Valores Singulares (SVD), se logró condensar la información de una matriz de utilidad dispersa (donde muchos usuarios no han calificado muchos restaurantes) en un espacio de factores latentes. El uso de 66 componentes permitió capturar la esencia de las preferencias de los usuarios (95% de la varianza), eliminando el "ruido" y facilitando la identificación de patrones de consumo que no son evidentes a simple vista.

Transición de Filtrado Colaborativo a Sistema Híbrido:
La actividad demostró que, aunque las calificaciones de los usuarios son valiosas, pueden ser limitadas. Al integrar variables categóricas (como el código de vestimenta, accesibilidad y precio) mediante One-Hot Encoding, transformamos un sistema de filtrado colaborativo puro en un modelo híbrido. Este enfoque es más robusto, ya que no solo considera "quién calificó qué", sino también las características intrínsecas del negocio, lo que suele traducirse en recomendaciones más precisas y personalizadas.

Importancia de la Similitud de Coseno:
El uso de la métrica de coseno resultó fundamental para comparar los vectores en el espacio latente. A diferencia de otras medidas, el coseno se enfoca en la orientación de los vectores (el perfil del restaurante) más que en su magnitud, lo que permite encontrar "vecinos cercanos" de manera efectiva incluso después de haber combinado datos de distinta naturaleza (calificaciones y metadatos OHE).

Impacto de la Limpieza de Datos:
El ejercicio inicial con 'Gorditas Dona Tota' subrayó una lección crítica en ciencia de datos: la calidad del modelo depende de la unicidad de las entidades. Sin la diferenciación de nombres y el manejo adecuado del encoding (Latin-1), el algoritmo habría fallado o generado sesgos, lo que demuestra que el preprocesamiento es tan importante como la elección del algoritmo mismo.

Conclusión General:
El sistema híbrido final ofrece una visión multidimensional del mercado restaurantero del ejercicio. Al buscar similares para "tacos de barbacoa enfrente del Tec", el modelo ahora es capaz de sugerir lugares que no solo gustan a los mismos usuarios, sino que comparten un rango de precio o una atmósfera similar, proporcionando una herramienta de valor real para la toma de decisiones del consumidor.



++++++++ Termina la sección de agregar texto. +++++++++++

# **++++ Fin de la Actividad de la semana: Sistema de Recomendación Híbrido +++**